In [1]:
import scipy.io as sio
import numpy as np
import pandas as pd

match_dir = '/Users/quinnmackay/Documents/GitHub/BICC_Holocene_LayerCount/matchfiles'

# Load and filter EDML_match.mat
edml_mp = sio.loadmat(f'{match_dir}/EDML_match.mat')['mp']
edml_mask = (edml_mp[:, 1] == 6) | (edml_mp[:, 1] == 7)
edml_filtered = edml_mp[edml_mask]

# Load and filter WDC_match.mat
wdc_mp = sio.loadmat(f'{match_dir}/WDC_match.mat')['mp']
wdc_mask = (wdc_mp[:, 1] == 6) | (wdc_mp[:, 1] == 7)
wdc_filtered = wdc_mp[wdc_mask]

In [2]:
#cores from each pole
GL_cores = ['GISP2', 'GRIP', 'NG1', 'NG2', 'NEEM']
AA_cores = ['EDML', 'WDC', 'DF', 'TALDICE', 'EDC']

In [3]:
#CREATED BY AI - just filtering the big CSV into what I need

# Load bipolar tiepoints — keep only Greenland–Antarctic (GL-AA) pairs.
bipolar_path = '/Users/quinnmackay/Documents/GitHub/IceTiepoint_Analysis/Network Analysis/big_table/big_table_all_tiepoints.xlsx'
bipolar_df = pd.read_excel(bipolar_path)

# -------------------------------------------------------------------
# Split into a dictionary keyed by core pair (e.g. "GISP2-EDML")
# Each value is a tidy DataFrame with columns: [core_A, core_B, reference, code]
# Only keep pairs that cross Greenland ↔ Antarctic (one GL core, one AA core).
# -------------------------------------------------------------------

# Only keep GL cores paired with WDC or EDML specifically
target_AA = ['EDML', 'WDC']

# Depth range filters (adjustable)
depth_range = {
    'EDML': (616.13, 673.28),
    'WDC':  (1823.0, 1962.0),
}

# Tiepoint codes to exclude (e.g. 'GISP2-EDML_3', 'GRIP-WDC_12')
exclude_codes = [
    'GISP2-WDC_162',
    'NG2-EDML_71'
]

def is_gl_target(pair_name):
    cores = pair_name.split('-')
    if len(cores) != 2:
        return False
    a, b = cores
    return (a in GL_cores and b in target_AA) or (a in target_AA and b in GL_cores)

# Collect all column names that look like "PAIR_suffix"
all_pair_columns = [col for col in bipolar_df.columns if '-' in col and '_' in col]

# Extract unique pair names
pair_names = sorted(set(
    col.rsplit('_', 1)[0] for col in all_pair_columns
))

# Keep only GL–(WDC or EDML) pairs
pair_names = [p for p in pair_names if is_gl_target(p)]

bipolar_pairs = {}
for pair in pair_names:
    # Columns belonging to this pair
    pair_cols = [c for c in all_pair_columns if c.startswith(pair + '_')]

    # Subset and drop rows that are all-NaN for this pair
    pair_df = bipolar_df[pair_cols].dropna(how='all').copy()

    if pair_df.empty:
        continue

    # Rename columns: strip the pair prefix so we get clean names
    # e.g. "GISP2-EDML_GISP2" → "GISP2", "GISP2-EDML_reference" → "reference"
    rename_map = {}
    for col in pair_df.columns:
        suffix = col[len(pair) + 1:]  # everything after "PAIR_"
        rename_map[col] = suffix

    pair_df.rename(columns=rename_map, inplace=True)

    # Apply depth-range filter for EDML or WDC if present
    for core, (dmin, dmax) in depth_range.items():
        if core in pair_df.columns:
            pair_df = pair_df[(pair_df[core] >= dmin) & (pair_df[core] <= dmax)]

    # Exclude specific tiepoint codes
    if 'code' in pair_df.columns and exclude_codes:
        pair_df = pair_df[~pair_df['code'].isin(exclude_codes)]

    if pair_df.empty:
        continue

    bipolar_pairs[pair] = pair_df

# Show what we have
for name, df in bipolar_pairs.items():
    print(f"{name}: {len(df)} tiepoints, columns = {list(df.columns)}")


GISP2-EDML: 5 tiepoints, columns = ['GISP2', 'EDML', 'reference', 'code']
GISP2-WDC: 18 tiepoints, columns = ['GISP2', 'WDC', 'reference', 'code']
GRIP-EDML: 5 tiepoints, columns = ['GRIP', 'EDML', 'reference', 'code']
GRIP-WDC: 16 tiepoints, columns = ['GRIP', 'WDC', 'reference', 'code']
NEEM-EDML: 5 tiepoints, columns = ['NEEM', 'EDML', 'reference', 'code']
NEEM-WDC: 5 tiepoints, columns = ['NEEM', 'WDC', 'reference', 'code']
NG2-EDML: 14 tiepoints, columns = ['NG2', 'EDML', 'reference', 'code']
NG2-WDC: 5 tiepoints, columns = ['NG2', 'WDC', 'reference', 'code']


In [4]:
for pair_pick in ['GISP2-WDC', 'GRIP-WDC', 'NG2-EDML']:
    if 'WDC' in pair_pick.split('-'):
        AA_core = 'WDC'
        AA_LC = wdc_filtered
    else:
        AA_core = 'EDML'
        AA_LC = edml_filtered

    bipolar_set = bipolar_pairs[pair_pick].copy()
    bipolar_set['total_layers_in_section'] = 'X'  # new column
    
    for i, row in enumerate(bipolar_set.itertuples(index=False)):
        if i == 0:
            continue

        antarctic_depth = getattr(row, AA_core)
        prev_antarctic_depth = bipolar_set[AA_core].iloc[i - 1]

        # Count layers between the two tiepoint depths
        mask = (AA_LC[:, 0] >= prev_antarctic_depth) & (AA_LC[:, 0] < antarctic_depth)
        total_layers_in_section = mask.sum()
        bipolar_set.iloc[i, -1] = total_layers_in_section  # last column
    
    bipolar_set.to_excel(f'/Users/quinnmackay/Desktop/layers_{pair_pick}.xlsx', index=False)
